# Extended Analysis

This notebook contains the extended grouped analysis for institution, familiarity, demographics, topic, AI familiarity, auditory sensitivity, and recording equipment.

In [ ]:
from pathlib import Path
from datetime import datetime
import re
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

DATA_DIR = Path("Data")
assert DATA_DIR.exists(), "Data directory does not exist."

# Note: reading questionary.xlsx requires openpyxl.
# If missing: uv add openpyxl && uv sync

In [ ]:
# Load trial-level data
result_files = sorted(DATA_DIR.glob("AI_VoiceTask_P*_results.csv"))
trial_df = pd.concat([pd.read_csv(f) for f in result_files], ignore_index=True)

trial_df["ParticipantID"] = trial_df["ParticipantID"].astype(str).str.strip().str.upper()
trial_df["ParticipantNum"] = trial_df["ParticipantID"].str.extract(r"(\d+)").astype(int)
trial_df["Institution"] = np.where(trial_df["ParticipantNum"].between(1, 25), "Institution 1", "Institution 2")

trial_df["Model"] = trial_df["Model"].astype(int)
trial_df["Speaker"] = trial_df["Speaker"].astype(int)
trial_df["X"] = trial_df["IsAI"].astype(int)
trial_df["Y"] = trial_df["Model"].isin([2, 3]).astype(int)
trial_df["Correct"] = (trial_df["X"] == trial_df["Y"]).astype(int)

trial_df["w"] = (
    trial_df["Confidence"].astype(str).str.extract(r"(\d+)").astype(float).div(100.0)
)
trial_df["wacc_component"] = np.where(trial_df["Correct"] == 1, trial_df["w"], 1 - trial_df["w"])
trial_df["Topic"] = ((trial_df["Artic"].astype(int) - 1) // 10 + 1).astype(int)

trial_df["Familiarity"] = np.where(
    ((trial_df["Institution"] == "Institution 2") & (trial_df["Speaker"] == 1))
    | ((trial_df["Institution"] == "Institution 1") & (trial_df["Speaker"].isin([3, 4]))),
    "Familiar",
    "Stranger",
)

speaker_meta = {
    1: {"SpeakerName": "shi", "SpeakerGender": "Female", "SpeakerDevice": "Good"},
    2: {"SpeakerName": "yl", "SpeakerGender": "Male", "SpeakerDevice": "Poor"},
    3: {"SpeakerName": "zy", "SpeakerGender": "Male", "SpeakerDevice": "Good"},
    4: {"SpeakerName": "xsh", "SpeakerGender": "Female", "SpeakerDevice": "Poor"},
}
trial_df["SpeakerName"] = trial_df["Speaker"].map(lambda x: speaker_meta[x]["SpeakerName"])
trial_df["SpeakerGender"] = trial_df["Speaker"].map(lambda x: speaker_meta[x]["SpeakerGender"])
trial_df["SpeakerDevice"] = trial_df["Speaker"].map(lambda x: speaker_meta[x]["SpeakerDevice"])

# Load questionnaire metadata
q_df = pd.read_excel(DATA_DIR / "questionary.xlsx")
participant_id_col = "4、您的受试者编号为？（*必答）"
participant_gender_col = "1、您的性别："
birth_year_col = "2、您的出生年份是？（*必答）"

q_df["ParticipantID"] = q_df[participant_id_col].astype(str).str.strip().str.upper()
q_df = q_df[q_df["ParticipantID"].str.match(r"^P\d+$", na=False)].copy()

gender_map = {1: "Male", 2: "Female", "1": "Male", "2": "Female", "男": "Male", "女": "Female"}
q_df["ParticipantGender"] = q_df[participant_gender_col].map(gender_map).fillna(q_df[participant_gender_col].astype(str))
q_df["BirthYear"] = pd.to_numeric(q_df[birth_year_col], errors="coerce")
q_df["ParticipantAge"] = 2026 - q_df["BirthYear"]

ai_cols = []
auditory_cols = []
for col in q_df.columns:
    m = re.match(r"^(\d+)、", str(col))
    if m:
        idx = int(m.group(1))
        if 6 <= idx <= 27:
            ai_cols.append(col)
        if 29 <= idx <= 33:
            auditory_cols.append(col)

q_df[ai_cols] = q_df[ai_cols].apply(pd.to_numeric, errors="coerce")
q_df[auditory_cols] = q_df[auditory_cols].apply(pd.to_numeric, errors="coerce")
q_df["AI_Familiarity"] = q_df[ai_cols].mean(axis=1)
q_df["Auditory_Sensitivity"] = q_df[auditory_cols].mean(axis=1)

participant_meta = q_df[[
    "ParticipantID", "ParticipantGender", "ParticipantAge", "AI_Familiarity", "Auditory_Sensitivity"
]].drop_duplicates("ParticipantID")

analysis_df = trial_df.merge(participant_meta, on="ParticipantID", how="left")

print("Trial rows:", len(analysis_df))
print("Participants in trials:", analysis_df["ParticipantID"].nunique())
print("Participants matched in questionnaire:", analysis_df["ParticipantGender"].notna().mean().round(3))
analysis_df.head(3)

In [ ]:
def _safe_mean(series: pd.Series) -> float:
    return np.nan if len(series) == 0 else series.mean()


def compute_metrics(df: pd.DataFrame) -> pd.Series:
    m1 = df[df["Model"] == 1]
    m2 = df[df["Model"] == 2]
    m3 = df[df["Model"] == 3]
    ai = df[df["Model"].isin([2, 3])]
    return pd.Series(
        {
            "n": len(df),
            "acc0": _safe_mean(df["Correct"]),
            "acc1": _safe_mean(m1["Correct"]),
            "acc2": _safe_mean(ai["Correct"]),
            "acc3": _safe_mean(m2["Correct"]),
            "acc4": _safe_mean(m3["Correct"]),
            "wacc0": _safe_mean(df["wacc_component"]),
        }
    )


def summarize_by(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    out = df.groupby(group_col, dropna=False).apply(compute_metrics).reset_index()
    return out.sort_values(group_col).reset_index(drop=True)

In [ ]:
# 1 Institution
res_institution = summarize_by(analysis_df, "Institution")[["Institution", "n", "acc0", "acc1", "acc3", "acc4", "wacc0"]]

# 2 Familiarity
res_familiarity = summarize_by(analysis_df, "Familiarity")[["Familiarity", "n", "acc0", "acc1", "acc3", "acc4", "wacc0"]]

# 3 Speaker
res_speaker = summarize_by(analysis_df, "Speaker")[["Speaker", "n", "acc0", "wacc0"]]

# 4 Speaker Gender
res_speaker_gender = summarize_by(analysis_df, "SpeakerGender")[["SpeakerGender", "n", "acc0", "wacc0"]]

# 5 Participant Gender
res_participant_gender = summarize_by(analysis_df.dropna(subset=["ParticipantGender"]), "ParticipantGender")[["ParticipantGender", "n", "acc0", "wacc0"]]

# 6 Participant Age
age_df = analysis_df.dropna(subset=["ParticipantAge"]).copy()
age_df["ParticipantAge"] = age_df["ParticipantAge"].astype(int)
res_participant_age = summarize_by(age_df, "ParticipantAge")[["ParticipantAge", "n", "acc0", "wacc0"]]

# 7 Topic
res_topic = summarize_by(analysis_df, "Topic")[["Topic", "n", "acc0", "wacc0"]]

# 8 AI Familiarity (binned)
ai_fam_df = analysis_df.dropna(subset=["AI_Familiarity"]).copy()
ai_fam_df["AI_Familiarity_Bin"] = pd.cut(
    ai_fam_df["AI_Familiarity"],
    bins=[0, 2.5, 3.5, 4.5, 5.1],
    labels=["Low", "Mid", "High", "Very High"],
    include_lowest=True,
)
res_ai_fam = summarize_by(ai_fam_df, "AI_Familiarity_Bin")[["AI_Familiarity_Bin", "n", "acc0", "wacc0"]]

# 9 Auditory Sensitivity (binned)
aud_df = analysis_df.dropna(subset=["Auditory_Sensitivity"]).copy()
aud_df["Auditory_Sensitivity_Bin"] = pd.cut(
    aud_df["Auditory_Sensitivity"],
    bins=[0, 2.5, 3.5, 4.5, 5.1],
    labels=["Low", "Mid", "High", "Very High"],
    include_lowest=True,
)
res_auditory = summarize_by(aud_df, "Auditory_Sensitivity_Bin")[["Auditory_Sensitivity_Bin", "n", "acc0", "wacc0"]]

# 10 Recording Equipment
res_equipment = summarize_by(analysis_df, "SpeakerDevice")[["SpeakerDevice", "n", "acc0", "wacc0"]]

print("1) Institution")
display(res_institution)
print("2) Familiarity")
display(res_familiarity)
print("3) Speaker")
display(res_speaker)
print("4) Speaker Gender")
display(res_speaker_gender)
print("5) Participant Gender")
display(res_participant_gender)
print("6) Participant Age")
display(res_participant_age)
print("7) Topic")
display(res_topic)
print("8) AI Familiarity")
display(res_ai_fam)
print("9) Auditory Sensitivity")
display(res_auditory)
print("10) Recording Equipment")
display(res_equipment)

In [ ]:
# Age histogram
fig, ax = plt.subplots(figsize=(7, 4))
age_values = age_df["ParticipantAge"].dropna().astype(int)
bins = np.arange(age_values.min(), age_values.max() + 2) - 0.5
sns.histplot(age_values, bins=bins, ax=ax, color="#4C72B0")
ax.set_title("Participant Age Distribution")
ax.set_xlabel("Age")
ax.set_ylabel("Count")
ax.set_xticks(sorted(age_values.unique()))
plt.tight_layout()

In [ ]:
# Optional: export extended analysis tables
result_tables = {
    "institution": res_institution,
    "familiarity": res_familiarity,
    "speaker": res_speaker,
    "speaker_gender": res_speaker_gender,
    "participant_gender": res_participant_gender,
    "participant_age": res_participant_age,
    "topic": res_topic,
    "ai_familiarity": res_ai_fam,
    "auditory_sensitivity": res_auditory,
    "recording_equipment": res_equipment,
}

base_output_dir = Path("outputs")
runs_dir = base_output_dir / "runs"
runs_dir.mkdir(parents=True, exist_ok=True)

run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = runs_dir / f"extended_{run_timestamp}"
output_dir.mkdir(parents=True, exist_ok=False)

for name, df in result_tables.items():
    df.to_csv(output_dir / f"{name}.csv", index=False)

latest_dir = base_output_dir / "latest_extended"
if latest_dir.exists() or latest_dir.is_symlink():
    if latest_dir.is_symlink() or latest_dir.is_file():
        latest_dir.unlink()
    else:
        shutil.rmtree(latest_dir)

try:
    latest_dir.symlink_to(output_dir.resolve(), target_is_directory=True)
    latest_note = f"latest symlink: {latest_dir.resolve()} -> {output_dir.resolve()}"
except OSError:
    shutil.copytree(output_dir, latest_dir)
    latest_note = f"latest copied directory: {latest_dir.resolve()}"

print(f"Exported extended results to: {output_dir.resolve()}")
print(latest_note)